<a href="https://colab.research.google.com/github/Victor-ro-dev/corn-pest-classifier/blob/main/treinamento_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌽 Classificador de Pragas do Milho — MobileNetV2 Híbrida (2 Estágios + Quantização TFLite)

Pipeline de treinamento supervisionado multiclasse de alta precisão para identificação de pragas na cultura do milho e integração com o **AgriRAG (WhatsApp / LangGraph)**.

### 🚀 Melhorias de Engenharia Aplicadas nesta Versão:
- **Arquitetura Híbrida (`mobilenet_custom.py`)**: Backbone `MobileNetV2` (ImageNet) + **Bloco Residual Invertido Customizado** de adaptação ao domínio agrícola + Cabeça com `BatchNormalization`, `Dropout(0.35)` e regularização `L2`.
- **Treinamento em 2 Estágios (`trainer.py`)**:
  1. **Fase 1 (Warmup)**: Treina o Bloco Residual Agrícola e a cabeça classificadora com o backbone congelado (`LR = 1e-3`).
  2. **Fase 2 (Fine-Tuning)**: Descongela seletivamente as últimas 35 camadas superiores da rede (mantendo `BatchNormalization` congelado) com `LR = 1e-4` e `clipnorm = 1.0`.
- **Balanceamento Automático (`class_weights`)**: Compensa diferenças na quantidade de fotos por classe diretamento dentro do `@tf.function`.
- **Data Augmentation de Campo (`data_loader.py`)**: Simula fotos reais tiradas por produtores no celular (variações de sol forte/sombra com `RandomBrightness` e `RandomContrast`, distância com `RandomZoom`, ângulo com `RandomRotation` e `RandomFlip`).
- **Integração Nativa com Data Flywheel (US03)**: Incorpora automaticamente as fotos confirmadas pelos produtores com *"Sim"* no WhatsApp.
- **Callbacks Nativos no GradientTape**: `ReduceLROnPlateau`, `EarlyStopping`, restauração automática dos melhores pesos (`best_weights`), gráficos de Loss/Acurácia (`training_curves.png`), Matriz de Confusão (`confusion_matrix.png`) e **Quantização Dinâmica TFLite**.

### 🎯 Mapeamento Oficial de Classes (Cultura do Milho):
1. **`lagarta_cartucho`** (*Spodoptera frugiperda*): Praga nº 1 — GBIF (larvas) + KaraAgro AI Maize.
2. **`lagarta_espiga`** (*Helicoverpa zea*): Decisiva para milho verde — GBIF (estágio larval).
3. **`vaquinha`** (*Diabrotica speciosa*): Besouro com dano foliar característico — GBIF (adultos).
4. **`milho_sadio`**: Classe negativa / controle sadio — PlantVillage / GBIF (*Zea mays*).

---

### 1. Configurando o Repositório no Colab (caso execute do zero)
Clona o repositório, entra no diretório do projeto e adiciona a pasta `src` ao path do Python:

In [ ]:
import os
import sys

# Clona o projeto se ainda não estiver clonado (ou atualiza via git pull)
if not os.path.exists('src'):
    if not os.path.exists('corn-pest-classifier'):
        !git clone https://github.com/Victor-ro-dev/corn-pest-classifier.git
    if os.path.exists('corn-pest-classifier'):
        %cd corn-pest-classifier
elif os.path.exists('.git'):
    !git pull

# Garante que a pasta src esteja acessível para importação em qualquer célula
if os.path.exists('src') and 'src' not in sys.path:
    sys.path.insert(0, os.path.abspath('src'))

print('Diretório ativo:', os.getcwd())
print('Python Path inclui src:', os.path.abspath('src') in sys.path)

### 2. Verificando o Acelerador de Hardware (GPU)
Certifique-se de que o ambiente de execução está configurado com GPU em: `Ambiente de execução` > `Alterar tipo de ambiente de execução` > `T4 GPU`.

In [ ]:
!nvidia-smi

### 3. Instalação das Dependências do Projeto

In [ ]:
!pip install -q -r requirements.txt

### 3.1. Injeção Automática dos Módulos Aprimorados na VM do Colab
Grava diretamente na máquina virtual do Colab as versões novas de `config.py`, `data_loader.py`, `mobilenet_custom.py`, `trainer.py`, `visualizator.py` e `main.py` (assim você **só precisa subir este arquivo `.ipynb` no Colab**, sem precisar subir os outros arquivos `.py`!):

In [ ]:
# Garante que os módulos Python aprimorados existam dentro da VM do Colab (mesmo sem git push prévio)
from pathlib import Path
Path('src/corn_pest_classifier').mkdir(parents=True, exist_ok=True)
Path('src/corn_pest_classifier/__init__.py').touch()

Path('src/corn_pest_classifier/config.py').write_text('import os\nfrom pathlib import Path\n\nBASE_DIR = Path(__file__).resolve().parent.parent.parent\n\nBATCH_SIZE = 32\nIMG_SIZE = (224, 224)\n\n# Treinamento em 2 Estágios (Transfer Learning + Fine-Tuning)\nWARMUP_EPOCHS = int(os.getenv("WARMUP_EPOCHS", "8"))\nFINETUNE_EPOCHS = int(os.getenv("FINETUNE_EPOCHS", "12"))\nEPOCHS = WARMUP_EPOCHS + FINETUNE_EPOCHS\n\nLEARNING_RATE = 1e-3\nFINETUNE_LEARNING_RATE = 1e-4\nWEIGHT_DECAY = 1e-4\nDROPOUT_RATE = 0.35\nEARLY_STOPPING_PATIENCE = 5\nLR_PATIENCE = 2\nUNFREEZE_TOP_LAYERS = 35\n\nDATASET_DIR = os.getenv("DATASET_DIR", str(BASE_DIR / "dataset"))\n# Diretório opcional do Data Flywheel (US03) com fotos confirmadas pelos produtores no WhatsApp\nFLYWHEEL_CONFIRMED_DIR = os.getenv(\n    "FLYWHEEL_CONFIRMED_DIR",\n    str(BASE_DIR.parent / "agri_rag" / "data" / "dataset_flywheel" / "confirmados"),\n)\n\n# Mapeamento oficial das classes do projeto (Escopo: Cultura do Milho)\nCLASS_NAMES = [\n    "lagarta_cartucho",  # Spodoptera frugiperda (GBIF larva + KaraAgro AI Maize)\n    "lagarta_espiga",    # Helicoverpa zea (GBIF larva + espiga)\n    "vaquinha",          # Diabrotica speciosa (GBIF adulto)\n    "milho_sadio",       # Classe negativa de controle (KaraAgro AI Maize / PlantVillage "healthy")\n]\nNUM_CLASSES = len(CLASS_NAMES)\n', encoding='utf-8')
Path('src/corn_pest_classifier/data_loader.py').write_text('import shutil\nimport sys\nfrom pathlib import Path\n\n# Garante a resolução dos módulos locais\n_pkg_dir = str(Path(__file__).resolve().parent)\nif _pkg_dir not in sys.path:\n    sys.path.insert(0, _pkg_dir)\n\nimport keras\nimport tensorflow as tf\n\ntry:\n    from corn_pest_classifier.config import (\n        BATCH_SIZE,\n        CLASS_NAMES,\n        DATASET_DIR,\n        FLYWHEEL_CONFIRMED_DIR,\n        IMG_SIZE,\n    )\nexcept ImportError:\n    from config import (\n        BATCH_SIZE,\n        CLASS_NAMES,\n        DATASET_DIR,\n        FLYWHEEL_CONFIRMED_DIR,\n        IMG_SIZE,\n    )\n\nVALID_IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}\n\n\ndef get_augmentation_pipeline() -> keras.Sequential:\n    """Pipeline realista de Data Augmentation simulando fotos de celular na lavoura (sol, sombra, zoom e ângulo)."""\n    return keras.Sequential(\n        [\n            keras.layers.RandomFlip("horizontal_and_vertical"),\n            keras.layers.RandomRotation(0.20),\n            keras.layers.RandomZoom(height_factor=(-0.15, 0.15), width_factor=(-0.15, 0.15)),\n            keras.layers.RandomTranslation(height_factor=0.10, width_factor=0.10),\n            keras.layers.RandomContrast(0.20),\n            keras.layers.RandomBrightness(factor=0.15, value_range=(0, 255)),\n        ],\n        name="Field_Photo_Augmentation",\n    )\n\n\ndef sync_flywheel_samples(dataset_dir: Path, flywheel_dir: Path | None = None) -> int:\n    """Importa fotos reais confirmadas pelos produtores no WhatsApp (Data Flywheel US03) para o dataset de treino."""\n    fw_path = Path(flywheel_dir or FLYWHEEL_CONFIRMED_DIR)\n    if not fw_path.exists():\n        return 0\n\n    alias_map = {\n        "spodoptera_frugiperda": "lagarta_cartucho",\n        "lagarta_cartucho": "lagarta_cartucho",\n        "helicoverpa_zea": "lagarta_espiga",\n        "lagarta_espiga": "lagarta_espiga",\n        "diabrotica_speciosa": "vaquinha",\n        "vaquinha": "vaquinha",\n        "milho_sadio": "milho_sadio",\n    }\n\n    copied = 0\n    for folder in fw_path.iterdir():\n        if not folder.is_dir():\n            continue\n        target_class = alias_map.get(folder.name.lower())\n        if not target_class:\n            continue\n\n        dest_dir = dataset_dir / target_class\n        dest_dir.mkdir(parents=True, exist_ok=True)\n\n        for img_file in folder.iterdir():\n            if img_file.suffix.lower() not in VALID_IMG_EXTENSIONS:\n                continue\n            # Ignora arquivos simulados de teste unitário (< 200 bytes)\n            if img_file.stat().st_size < 200:\n                continue\n            dest_file = dest_dir / f"flywheel_{img_file.name}"\n            if not dest_file.exists():\n                shutil.copy2(img_file, dest_file)\n                copied += 1\n\n    if copied > 0:\n        print(f"[Data Flywheel US03] +{copied} novas fotos confirmadas no campo incorporadas ao treino!")\n    return copied\n\n\ndef compute_class_weights(dataset_dir: Path, active_classes: list[str]) -> dict[int, float]:\n    """Calcula pesos balanceados inversamente proporcionais à frequência de cada classe:\n    w_c = Total / (Num_Classes * Count_c)\n    Evita que a CNN fique enviesada para a classe majoritária.\n    """\n    counts: dict[int, int] = {}\n    for idx, cls_name in enumerate(active_classes):\n        cls_folder = dataset_dir / cls_name\n        if cls_folder.exists():\n            imgs = [\n                f\n                for f in cls_folder.iterdir()\n                if f.is_file() and f.suffix.lower() in VALID_IMG_EXTENSIONS\n            ]\n            counts[idx] = max(len(imgs), 1)\n        else:\n            counts[idx] = 1\n\n    total_samples = sum(counts.values())\n    num_classes = max(len(active_classes), 1)\n\n    weights = {\n        idx: round(total_samples / (num_classes * count), 4)\n        for idx, count in counts.items()\n    }\n    print("Balanceamento de Classes (Class Weights):")\n    for idx, cls_name in enumerate(active_classes):\n        print(f"  - [{idx}] {cls_name}: {counts[idx]} fotos (peso = {weights[idx]:.3f}x)")\n    return weights\n\n\ndef build_dataset(dataset_dir: str | None = None):\n    """Constrói os pipelines `train_ds` e `val_ds` otimizados com cache, prefetch e pesos de classe."""\n    target_path = Path(dataset_dir or DATASET_DIR)\n\n    # 1. Sincroniza automaticamente fotos validadas no Data Flywheel (US03), se existirem\n    sync_flywheel_samples(target_path)\n\n    # 2. Filtra e carrega estritamente as classes oficiais do projeto que existem no diretório\n    classes_oficiais_presentes = [\n        c for c in CLASS_NAMES if (target_path / c).is_dir()\n    ]\n    filtro_classes = classes_oficiais_presentes if classes_oficiais_presentes else None\n\n    train_ds, val_ds = keras.utils.image_dataset_from_directory(\n        str(target_path),\n        class_names=filtro_classes,\n        validation_split=0.2,\n        subset="both",\n        seed=42,\n        image_size=IMG_SIZE,\n        batch_size=BATCH_SIZE,\n    )\n\n    class_names = list(getattr(train_ds, "class_names", []))\n    print(f"Classes ativas no dataset ({len(class_names)}): {class_names}")\n    faltantes = [c for c in CLASS_NAMES if c not in class_names]\n    if faltantes:\n        print(\n            f"[AVISO] Classes esperadas do projeto ausentes no diretório: {faltantes}\\n"\n            f"Classes oficiais esperadas ({len(CLASS_NAMES)}): {CLASS_NAMES}"\n        )\n\n    # 3. Calcula os pesos de balanceamento para compensar desbalanceamento de amostras\n    class_weights = compute_class_weights(target_path, class_names) if class_names else {}\n\n    # 4. Data Augmentation de Campo + Pipeline de Alta Performance (AUTOTUNE)\n    data_augmentation = get_augmentation_pipeline()\n\n    train_ds = train_ds.shuffle(buffer_size=512, seed=42)\n    train_ds = train_ds.map(\n        lambda x, y: (data_augmentation(x, training=True), y),\n        num_parallel_calls=tf.data.AUTOTUNE,\n    )\n    train_ds = train_ds.prefetch(tf.data.AUTOTUNE)\n    val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)\n\n    # Preserva atributos úteis nos datasets retornados\n    train_ds.class_names = class_names\n    val_ds.class_names = class_names\n    train_ds.class_weights = class_weights\n\n    return train_ds, val_ds\n', encoding='utf-8')
Path('src/corn_pest_classifier/mobilenet_custom.py').write_text('from tensorflow import keras\n\n\ndef inverted_residual_block(\n    x,\n    expand_filters: int,\n    squeeze_filters: int,\n    stride: int = 1,\n    weight_decay: float = 1e-4,\n    block_id: str = "b",\n):\n    """Bloco Residual Invertido (MobileNetV2):\n    1. Expansão Pointwise (Conv 1x1 + BN + ReLU6)\n    2. Filtragem Espacial Depthwise (DepthwiseConv 3x3 + BN + ReLU6)\n    3. Projeção Linear / Compressão (Conv 1x1 + BN sem ativação não-linear)\n    4. Skip Connection se stride == 1 e canais de entrada == saída.\n    """\n    reg = keras.regularizers.l2(weight_decay)\n    in_channels = x.shape[-1]\n\n    # 1. Expansão (só expande se expand_filters > in_channels)\n    if expand_filters != in_channels:\n        m = keras.layers.Conv2D(\n            expand_filters,\n            kernel_size=1,\n            padding="same",\n            use_bias=False,\n            kernel_regularizer=reg,\n            name=f"{block_id}_expand_conv",\n        )(x)\n        m = keras.layers.BatchNormalization(name=f"{block_id}_expand_bn")(m)\n        m = keras.layers.ReLU(max_value=6.0, name=f"{block_id}_expand_relu6")(m)\n    else:\n        m = x\n\n    # 2. Depthwise Convolution 3x3\n    m = keras.layers.DepthwiseConv2D(\n        kernel_size=3,\n        strides=stride,\n        padding="same",\n        use_bias=False,\n        depthwise_regularizer=reg,\n        name=f"{block_id}_depthwise",\n    )(m)\n    m = keras.layers.BatchNormalization(name=f"{block_id}_depthwise_bn")(m)\n    m = keras.layers.ReLU(max_value=6.0, name=f"{block_id}_depthwise_relu6")(m)\n\n    # 3. Projeção Linear (Gargalo Estreito - Linear Bottleneck)\n    m = keras.layers.Conv2D(\n        squeeze_filters,\n        kernel_size=1,\n        padding="same",\n        use_bias=False,\n        kernel_regularizer=reg,\n        name=f"{block_id}_project_conv",\n    )(m)\n    m = keras.layers.BatchNormalization(name=f"{block_id}_project_bn")(m)\n\n    # 4. Conexão Residual (Atalho de Identidade)\n    if in_channels == squeeze_filters and stride == 1:\n        return keras.layers.Add(name=f"{block_id}_add")([x, m])\n    return m\n\n\ndef build_custom_mobilenet(\n    input_shape: tuple[int, int, int] = (224, 224, 3),\n    num_classes: int = 4,\n    use_imagenet_backbone: bool = True,\n    dropout_rate: float = 0.35,\n    weight_decay: float = 1e-4,\n) -> keras.Model:\n    """Constrói a MobileNetV2 Híbrida para classificação de pragas do milho:\n    - Combina Transfer Learning (pesos ImageNet congelados na Fase 1) + Bloco Residual Invertido Customizado\n      de refinamento de domínio agrícola + Cabeça regularizada (BatchNorm + Dropout + L2).\n    - Caso esteja sem internet para baixar os pesos da ImageNet, constrói a pilha profunda customizada de 7 estágios.\n    """\n    reg = keras.regularizers.l2(weight_decay)\n    inputs = keras.Input(shape=input_shape, name="input_image")\n\n    # Pré-processamento nativo MobileNetV2: normaliza pixels de [0, 255] para [-1, 1]\n    x = keras.applications.mobilenet_v2.preprocess_input(inputs)\n\n    backbone_loaded = False\n    if use_imagenet_backbone:\n        try:\n            base_backbone = keras.applications.MobileNetV2(\n                input_shape=input_shape,\n                include_top=False,\n                weights="imagenet",\n                alpha=1.0,\n            )\n            # Fase 1 (Warmup): Congela o extrator de características pré-treinado\n            base_backbone.trainable = False\n            x = base_backbone(x, training=False)\n            # Bloco Residual Invertido Customizado de Adaptação de Domínio Agrícola\n            x = inverted_residual_block(\n                x,\n                expand_filters=512,\n                squeeze_filters=256,\n                stride=1,\n                weight_decay=weight_decay,\n                block_id="agro_neck",\n            )\n            backbone_loaded = True\n            print("✓ Backbone MobileNetV2 (ImageNet) + Bloco Residual Agrícola carregados com sucesso!")\n        except Exception as e:\n            print(f"[Aviso] Não foi possível baixar pesos ImageNet ({e}). Usando pilha profunda customizada.")\n\n    if not backbone_loaded:\n        # Stem Inicial com BatchNormalization e ReLU6\n        x = keras.layers.Conv2D(\n            32,\n            kernel_size=3,\n            strides=2,\n            padding="same",\n            use_bias=False,\n            kernel_regularizer=reg,\n            name="stem_conv",\n        )(x)\n        x = keras.layers.BatchNormalization(name="stem_bn")(x)\n        x = keras.layers.ReLU(max_value=6.0, name="stem_relu6")(x)\n\n        # Pilha profunda de Blocos Residuais Invertidos (7 estágios progressivos)\n        x = inverted_residual_block(x, expand_filters=32, squeeze_filters=16, stride=1, weight_decay=weight_decay, block_id="b1")\n        x = inverted_residual_block(x, expand_filters=96, squeeze_filters=24, stride=2, weight_decay=weight_decay, block_id="b2_1")\n        x = inverted_residual_block(x, expand_filters=144, squeeze_filters=24, stride=1, weight_decay=weight_decay, block_id="b2_2")\n        x = inverted_residual_block(x, expand_filters=144, squeeze_filters=32, stride=2, weight_decay=weight_decay, block_id="b3_1")\n        x = inverted_residual_block(x, expand_filters=192, squeeze_filters=32, stride=1, weight_decay=weight_decay, block_id="b3_2")\n        x = inverted_residual_block(x, expand_filters=192, squeeze_filters=64, stride=2, weight_decay=weight_decay, block_id="b4_1")\n        x = inverted_residual_block(x, expand_filters=384, squeeze_filters=96, stride=1, weight_decay=weight_decay, block_id="b5_1")\n        x = inverted_residual_block(x, expand_filters=576, squeeze_filters=160, stride=2, weight_decay=weight_decay, block_id="b6_1")\n\n        # Expansão final de features antes do pooling\n        x = keras.layers.Conv2D(\n            320,\n            kernel_size=1,\n            padding="same",\n            use_bias=False,\n            kernel_regularizer=reg,\n            name="head_conv1x1",\n        )(x)\n        x = keras.layers.BatchNormalization(name="head_bn")(x)\n        x = keras.layers.ReLU(max_value=6.0, name="head_relu6")(x)\n\n    # Cabeça de Classificação Regularizada contra Overfitting\n    x = keras.layers.GlobalAveragePooling2D(name="global_avg_pool")(x)\n    x = keras.layers.BatchNormalization(name="classifier_bn")(x)\n    x = keras.layers.Dropout(dropout_rate, name="classifier_dropout_1")(x)\n    x = keras.layers.Dense(\n        128,\n        activation="relu",\n        kernel_regularizer=reg,\n        name="classifier_dense_128",\n    )(x)\n    x = keras.layers.Dropout(dropout_rate * 0.5, name="classifier_dropout_2")(x)\n    outputs = keras.layers.Dense(\n        num_classes,\n        activation="softmax",\n        name="pest_probabilities",\n    )(x)\n\n    return keras.Model(inputs, outputs, name="Custom_MobileNetV2_Milho")\n\n\ndef unfreeze_for_finetuning(model: keras.Model, unfreeze_top_layers: int = 35) -> int:\n    """Fase 2 (Fine-Tuning): Descongela seletivamente as últimas `unfreeze_top_layers` camadas do backbone\n    mantendo camadas de BatchNormalization congeladas para preservar as estatísticas móveis.\n    """\n    unfrozen_count = 0\n    for layer in model.layers:\n        if isinstance(layer, keras.Model) and "mobilenetv2" in layer.name.lower():\n            layer.trainable = True\n            total_sublayers = len(layer.layers)\n            cutoff = max(0, total_sublayers - unfreeze_top_layers)\n            for idx, sublayer in enumerate(layer.layers):\n                if idx < cutoff or isinstance(sublayer, keras.layers.BatchNormalization):\n                    sublayer.trainable = False\n                else:\n                    sublayer.trainable = True\n                    unfrozen_count += 1\n    if unfrozen_count > 0:\n        print(f"🔓 [Fine-Tuning] {unfrozen_count} camadas superiores do backbone descongeladas (BatchNorm preservado)!")\n    return unfrozen_count\n', encoding='utf-8')
Path('src/corn_pest_classifier/trainer.py').write_text('import time\nfrom typing import Any\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport tensorflow as tf\nfrom tensorflow import keras\n\n\nclass CustomTrainer:\n    """Motor de treinamento customizado com @tf.function, GradientTape, Class Weights,\n    ReduceLROnPlateau, Early Stopping, Restauração dos Melhores Pesos e Relatório de Métricas.\n    """\n\n    def __init__(\n        self,\n        model: keras.Model,\n        learning_rate: float = 1e-3,\n        class_weights: dict[int, float] | None = None,\n        early_stopping_patience: int = 5,\n        lr_patience: int = 2,\n        min_lr: float = 1e-6,\n    ):\n        self.model = model\n        self.loss_fn = keras.losses.SparseCategoricalCrossentropy(reduction="none")\n        self.optimizer = keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=1.0)\n\n        self.early_stopping_patience = early_stopping_patience\n        self.lr_patience = lr_patience\n        self.min_lr = min_lr\n\n        # Tensor de pesos de classe para balanceamento dentro do @tf.function\n        num_outputs = model.output_shape[-1] or 4\n        if class_weights:\n            weights_list = [float(class_weights.get(i, 1.0)) for i in range(num_outputs)]\n        else:\n            weights_list = [1.0] * num_outputs\n        self.class_weight_tensor = tf.constant(weights_list, dtype=tf.float32)\n\n        # Métricas acumuladas por época (Média real de todos os batches)\n        self.train_loss_metric = keras.metrics.Mean(name="train_loss")\n        self.val_loss_metric = keras.metrics.Mean(name="val_loss")\n        self.train_acc_metric = keras.metrics.SparseCategoricalAccuracy(name="train_acc")\n        self.val_acc_metric = keras.metrics.SparseCategoricalAccuracy(name="val_acc")\n\n        # Estado global dos melhores pesos entre estágios\n        self.best_val_loss = float("inf")\n        self.best_val_acc = 0.0\n        self.best_weights: list[Any] | None = None\n        self.history: dict[str, list[float]] = {\n            "train_loss": [],\n            "val_loss": [],\n            "train_acc": [],\n            "val_acc": [],\n            "lr": [],\n        }\n\n    def set_learning_rate(self, new_lr: float) -> None:\n        """Atualiza dinamicamente o Learning Rate do otimizador."""\n        self.optimizer.learning_rate.assign(float(new_lr))\n\n    @tf.function\n    def train_step(self, x_batch, y_batch):\n        y_int = tf.cast(y_batch, tf.int32)\n        sample_weights = tf.gather(self.class_weight_tensor, y_int)\n\n        with tf.GradientTape() as tape:\n            probs = self.model(x_batch, training=True)\n            per_sample_loss = self.loss_fn(y_int, probs)\n            weighted_loss = tf.reduce_mean(per_sample_loss * sample_weights)\n            if self.model.losses:\n                total_loss = weighted_loss + tf.add_n(self.model.losses)\n            else:\n                total_loss = weighted_loss\n\n        grads = tape.gradient(total_loss, self.model.trainable_weights)\n        self.optimizer.apply_gradients(zip(grads, self.model.trainable_weights))\n\n        self.train_loss_metric.update_state(total_loss)\n        self.train_acc_metric.update_state(y_int, probs)\n        return total_loss\n\n    @tf.function\n    def test_step(self, x_batch, y_batch):\n        y_int = tf.cast(y_batch, tf.int32)\n        val_probs = self.model(x_batch, training=False)\n        per_sample_loss = self.loss_fn(y_int, val_probs)\n        val_loss = tf.reduce_mean(per_sample_loss)\n\n        self.val_loss_metric.update_state(val_loss)\n        self.val_acc_metric.update_state(y_int, val_probs)\n        return val_loss\n\n    def fit(\n        self,\n        train_ds,\n        val_ds,\n        epochs: int,\n        stage_name: str = "Treinamento",\n    ) -> dict[str, list[float]]:\n        """Executa o loop de épocas com ReduceLROnPlateau, EarlyStopping e salvamento dos melhores pesos."""\n        # Se o dataset já trouxe class_weights calculados, atualiza o tensor\n        ds_weights = getattr(train_ds, "class_weights", None)\n        if ds_weights:\n            num_outputs = self.model.output_shape[-1] or len(ds_weights)\n            w_list = [float(ds_weights.get(i, 1.0)) for i in range(num_outputs)]\n            self.class_weight_tensor = tf.constant(w_list, dtype=tf.float32)\n\n        epochs_without_improvement = 0\n        lr_wait = 0\n\n        for epoch in range(epochs):\n            start_time = time.time()\n            self.train_loss_metric.reset_state()\n            self.train_acc_metric.reset_state()\n            self.val_loss_metric.reset_state()\n            self.val_acc_metric.reset_state()\n\n            for x_batch_train, y_batch_train in train_ds:\n                self.train_step(x_batch_train, y_batch_train)\n\n            for x_batch_val, y_batch_val in val_ds:\n                self.test_step(x_batch_val, y_batch_val)\n\n            train_loss = float(self.train_loss_metric.result())\n            train_acc = float(self.train_acc_metric.result())\n            val_loss = float(self.val_loss_metric.result())\n            val_acc = float(self.val_acc_metric.result())\n            current_lr = float(self.optimizer.learning_rate.numpy())\n\n            self.history["train_loss"].append(train_loss)\n            self.history["val_loss"].append(val_loss)\n            self.history["train_acc"].append(train_acc)\n            self.history["val_acc"].append(val_acc)\n            self.history["lr"].append(current_lr)\n\n            elapsed = time.time() - start_time\n\n            # Verifica se houve melhora na validação (menor val_loss ou maior val_acc com empate de loss)\n            improved = False\n            if val_loss < (self.best_val_loss - 1e-4) or (\n                val_acc > self.best_val_acc and val_loss <= self.best_val_loss * 1.02\n            ):\n                self.best_val_loss = min(val_loss, self.best_val_loss)\n                self.best_val_acc = max(val_acc, self.best_val_acc)\n                self.best_weights = self.model.get_weights()\n                improved = True\n                epochs_without_improvement = 0\n                lr_wait = 0\n                try:\n                    self.model.save("best_milho_pragas.keras")\n                except Exception:\n                    pass\n            else:\n                epochs_without_improvement += 1\n                lr_wait += 1\n\n            badge = "★ [MELHOR MODELO SALVO]" if improved else ""\n            print(\n                f"[{stage_name}] Época {epoch + 1:02d}/{epochs:02d} ({elapsed:.1f}s) | "\n                f"LR: {current_lr:.1e} | "\n                f"Treino Loss: {train_loss:.4f} - Acc: {train_acc:.2%} | "\n                f"Val Loss: {val_loss:.4f} - Acc: {val_acc:.2%} {badge}"\n            )\n\n            # ReduceLROnPlateau\n            if lr_wait >= self.lr_patience:\n                new_lr = max(current_lr * 0.5, self.min_lr)\n                if new_lr < current_lr:\n                    self.set_learning_rate(new_lr)\n                    print(f"  📉 [ReduceLROnPlateau] Reduzindo Learning Rate para {new_lr:.1e}")\n                lr_wait = 0\n\n            # Early Stopping\n            if epochs_without_improvement >= self.early_stopping_patience:\n                print(\n                    f"  🛑 [Early Stopping] Sem melhora por {self.early_stopping_patience} épocas seguidas. Encerrando estágio."\n                )\n                break\n\n        # Restaura os melhores pesos validados\n        if self.best_weights is not None:\n            self.model.set_weights(self.best_weights)\n            print(\n                f"✓ Melhores pesos restaurados (Melhor Val Loss: {self.best_val_loss:.4f} | Melhor Val Acc: {self.best_val_acc:.2%})"\n            )\n\n        return self.history\n\n    def plot_training_curves(\n        self,\n        save_path: str = "training_curves.png",\n        warmup_epochs_ran: int | None = None,\n    ) -> None:\n        """Plota e salva as curvas de Loss e Acurácia de Treino vs Validação."""\n        if not self.history["train_loss"]:\n            return\n\n        epochs_range = range(1, len(self.history["train_loss"]) + 1)\n        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))\n\n        # 1. Curva de Loss\n        ax1.plot(epochs_range, self.history["train_loss"], "o-", label="Treino Loss", color="#1f77b4")\n        ax1.plot(epochs_range, self.history["val_loss"], "s-", label="Validação Loss", color="#d62728")\n        if warmup_epochs_ran and warmup_epochs_ran < len(epochs_range):\n            ax1.axvline(x=warmup_epochs_ran + 0.5, color="gray", linestyle="--", label="Início Fine-Tuning")\n        ax1.set_title("Evolução da Perda (Loss)", fontsize=12, fontweight="bold")\n        ax1.set_xlabel("Época")\n        ax1.set_ylabel("Loss")\n        ax1.grid(True, alpha=0.3)\n        ax1.legend()\n\n        # 2. Curva de Acurácia\n        ax2.plot(epochs_range, self.history["train_acc"], "o-", label="Treino Acurácia", color="#2ca02c")\n        ax2.plot(epochs_range, self.history["val_acc"], "s-", label="Validação Acurácia", color="#ff7f0e")\n        if warmup_epochs_ran and warmup_epochs_ran < len(epochs_range):\n            ax2.axvline(x=warmup_epochs_ran + 0.5, color="gray", linestyle="--", label="Início Fine-Tuning")\n        ax2.set_title("Evolução da Acurácia", fontsize=12, fontweight="bold")\n        ax2.set_xlabel("Época")\n        ax2.set_ylabel("Acurácia")\n        ax2.grid(True, alpha=0.3)\n        ax2.legend()\n\n        plt.tight_layout()\n        plt.savefig(save_path, dpi=150)\n        plt.close(fig)\n        print(f"[OK] Curvas de treinamento salvas em: {save_path}")\n\n    def evaluate_per_class(\n        self,\n        val_ds,\n        class_names: list[str],\n        save_path: str = "confusion_matrix.png",\n    ) -> dict[str, Any]:\n        """Gera a Matriz de Confusão e calcula Precisão, Recall e F1-Score por classe na validação."""\n        y_true_all: list[int] = []\n        y_pred_all: list[int] = []\n\n        for x_batch, y_batch in val_ds:\n            probs = self.model(x_batch, training=False).numpy()\n            preds = np.argmax(probs, axis=1)\n            y_true_all.extend(y_batch.numpy().astype(int).tolist())\n            y_pred_all.extend(preds.astype(int).tolist())\n\n        num_classes = max(len(class_names), 1)\n        cm = np.zeros((num_classes, num_classes), dtype=int)\n        for t, p in zip(y_true_all, y_pred_all):\n            if 0 <= t < num_classes and 0 <= p < num_classes:\n                cm[t, p] += 1\n\n        print("\\n=== RELATÓRIO DE DESEMPENHO POR CLASSE (VALIDAÇÃO) ===")\n        metrics_by_class: dict[str, dict[str, float]] = {}\n        for i, cls_name in enumerate(class_names):\n            tp = int(cm[i, i])\n            fp = int(np.sum(cm[:, i]) - tp)\n            fn = int(np.sum(cm[i, :]) - tp)\n            precision = tp / max(tp + fp, 1)\n            recall = tp / max(tp + fn, 1)\n            f1 = 2 * precision * recall / max(precision + recall, 1e-8)\n            metrics_by_class[cls_name] = {\n                "precision": round(precision, 4),\n                "recall": round(recall, 4),\n                "f1": round(f1, 4),\n                "support": int(np.sum(cm[i, :])),\n            }\n            print(\n                f"  - {cls_name:<18} | Precisão: {precision:.1%} | Recall: {recall:.1%} | "\n                f"F1-Score: {f1:.1%} ({tp}/{int(np.sum(cm[i, :]))} acertos)"\n            )\n\n        # Plota a Matriz de Confusão\n        fig, ax = plt.subplots(figsize=(7, 6))\n        im = ax.imshow(cm, interpolation="nearest", cmap="Greens")\n        fig.colorbar(im, ax=ax)\n        ax.set(\n            xticks=np.arange(num_classes),\n            yticks=np.arange(num_classes),\n            xticklabels=class_names,\n            yticklabels=class_names,\n            title="Matriz de Confusão - Validação (MobileNetV2 Milho)",\n            ylabel="Classe Real",\n            xlabel="Classe Predita",\n        )\n        plt.setp(ax.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")\n\n        thresh = cm.max() / 2.0 if cm.size > 0 else 0\n        for i in range(num_classes):\n            for j in range(num_classes):\n                ax.text(\n                    j,\n                    i,\n                    format(cm[i, j], "d"),\n                    ha="center",\n                    va="center",\n                    color="white" if cm[i, j] > thresh else "black",\n                    fontweight="bold",\n                )\n\n        plt.tight_layout()\n        plt.savefig(save_path, dpi=150)\n        plt.close(fig)\n        print(f"[OK] Matriz de Confusão salva em: {save_path}")\n\n        return {"confusion_matrix": cm.tolist(), "per_class": metrics_by_class}\n', encoding='utf-8')
Path('src/corn_pest_classifier/visualizator.py').write_text('import sys\nfrom pathlib import Path\n\n# Adiciona o diretório atual ao path para compatibilidade de importação\n_pkg_dir = str(Path(__file__).resolve().parent)\nif _pkg_dir not in sys.path:\n    sys.path.insert(0, _pkg_dir)\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport tensorflow as tf\nfrom tensorflow import keras\n\ntry:\n    from corn_pest_classifier.data_loader import build_dataset, get_augmentation_pipeline\nexcept ImportError:\n    from data_loader import build_dataset, get_augmentation_pipeline\n\n\ndef visualizar_lote():\n    """\n    Carrega o dataset e gera uma visualização gráfica de 9 imagens do lote,\n    salvando a figura em disco e abrindo a janela do matplotlib.\n    """\n    print("Carregando datasets...")\n    train_ds, val_ds = build_dataset()\n\n    class_names = getattr(train_ds, "class_names", [])\n    print(f"Classes detectadas ({len(class_names)}): {class_names}")\n\n    # Pega exatamente 1 lote (batch) de treino\n    for images, labels in train_ds.take(1):\n        print("\\n[INFO DO LOTE]")\n        print(f" - Formato do tensor (Batch, Altura, Largura, Canais): {images.shape}")\n        print(f" - Tipo de dado: {images.dtype}")\n        print(f" - Valor min do pixel: {float(tf.reduce_min(images)):.1f}, max: {float(tf.reduce_max(images)):.1f}")\n\n        # -------------------------------------------------------------\n        # 1. Grade 3x3 com 9 imagens do lote\n        # -------------------------------------------------------------\n        fig, axes = plt.subplots(3, 3, figsize=(11, 11))\n        fig.suptitle("Amostras do Lote de Treinamento", fontsize=16, fontweight="bold")\n\n        qtd_imagens = min(9, len(images))\n        for i in range(qtd_imagens):\n            ax = axes[i // 3, i % 3]\n            img = images[i].numpy()\n\n            # Ajusta valores caso estejam normalizados entre 0 e 1 ou em 0 e 255\n            if img.max() <= 1.0:\n                img_display = (img * 255).astype(np.uint8)\n            else:\n                img_display = img.astype(np.uint8)\n\n            label_idx = int(labels[i])\n            nome_classe = class_names[label_idx] if label_idx < len(class_names) else f"Classe {label_idx}"\n\n            ax.imshow(img_display)\n            ax.set_title(f"{nome_classe} (ID: {label_idx})", fontsize=11, fontweight="medium")\n            ax.axis("off")\n\n        # Desativa eixos excedentes caso o batch tenha menos de 9 fotos\n        for j in range(qtd_imagens, 9):\n            axes[j // 3, j % 3].axis("off")\n\n        plt.tight_layout()\n        caminho_salvar_lote = "preview_batch.png"\n        plt.savefig(caminho_salvar_lote, dpi=150)\n        print(f"\\n[OK] Imagem do lote salva em: {caminho_salvar_lote}")\n\n        # -------------------------------------------------------------\n        # 2. Demonstração do Data Augmentation de Campo na 1ª imagem\n        # -------------------------------------------------------------\n        data_augmentation = get_augmentation_pipeline()\n\n        fig_aug, axes_aug = plt.subplots(3, 3, figsize=(11, 11))\n        fig_aug.suptitle("Efeito do Data Augmentation na mesma imagem (9 variações)", fontsize=14, fontweight="bold")\n\n        primeira_imagem = images[0]\n        for i in range(9):\n            ax = axes_aug[i // 3, i % 3]\n            img_aug = data_augmentation(tf.expand_dims(primeira_imagem, 0), training=True)[0].numpy()\n\n            if img_aug.max() <= 1.0:\n                img_display = (img_aug * 255).astype(np.uint8)\n            else:\n                img_display = img_aug.astype(np.uint8)\n\n            ax.imshow(img_display)\n            ax.set_title(f"Variação #{i + 1}", fontsize=10)\n            ax.axis("off")\n\n        plt.tight_layout()\n        caminho_salvar_aug = "preview_augmentation.png"\n        plt.savefig(caminho_salvar_aug, dpi=150)\n        print(f"[OK] Amostra de Data Augmentation salva em: {caminho_salvar_aug}")\n\n        # Exibe na tela (caso haja interface gráfica ativa)\n        plt.show()\n        break\n\n\nif __name__ == "__main__":\n    visualizar_lote()\n', encoding='utf-8')
Path('src/corn_pest_classifier/main.py').write_text('import json\nimport sys\nfrom pathlib import Path\n\n# Garante compatibilidade de imports em qualquer ambiente (local, Colab, CLI)\n_pkg_dir = str(Path(__file__).resolve().parent)\nif _pkg_dir not in sys.path:\n    sys.path.insert(0, _pkg_dir)\n\nimport tensorflow as tf\n\ntry:\n    from corn_pest_classifier.config import (\n        DROPOUT_RATE,\n        EARLY_STOPPING_PATIENCE,\n        FINETUNE_EPOCHS,\n        FINETUNE_LEARNING_RATE,\n        LEARNING_RATE,\n        LR_PATIENCE,\n        NUM_CLASSES,\n        UNFREEZE_TOP_LAYERS,\n        WARMUP_EPOCHS,\n        WEIGHT_DECAY,\n    )\n    from corn_pest_classifier.data_loader import build_dataset\n    from corn_pest_classifier.mobilenet_custom import (\n        build_custom_mobilenet,\n        unfreeze_for_finetuning,\n    )\n    from corn_pest_classifier.trainer import CustomTrainer\nexcept ImportError:\n    from config import (\n        DROPOUT_RATE,\n        EARLY_STOPPING_PATIENCE,\n        FINETUNE_EPOCHS,\n        FINETUNE_LEARNING_RATE,\n        LEARNING_RATE,\n        LR_PATIENCE,\n        NUM_CLASSES,\n        UNFREEZE_TOP_LAYERS,\n        WARMUP_EPOCHS,\n        WEIGHT_DECAY,\n    )\n    from data_loader import build_dataset\n    from mobilenet_custom import build_custom_mobilenet, unfreeze_for_finetuning\n    from trainer import CustomTrainer\n\n\ndef export_quantized_tflite(\n    model: tf.keras.Model,\n    output_path: str = "milho_pragas_model.tflite",\n    quantize: bool = True,\n) -> Path:\n    """Converte o modelo Keras validado para TFLite com quantização dinâmica de pesos."""\n    converter = tf.lite.TFLiteConverter.from_keras_model(model)\n    if quantize:\n        converter.optimizations = [tf.lite.Optimize.DEFAULT]\n    tflite_model = converter.convert()\n\n    out_file = Path(output_path)\n    out_file.write_bytes(tflite_model)\n    size_kb = out_file.stat().st_size / 1024.0\n    print(f"✓ Modelo TFLite exportado com sucesso: \'{out_file}\' ({size_kb:.1f} KB)")\n    return out_file\n\n\ndef main():\n    print("1. Construindo o Dataset com Balanceamento de Classes e Data Augmentation de Campo...")\n    train_ds, val_ds = build_dataset()\n\n    class_names = list(getattr(train_ds, "class_names", []))\n    class_weights = getattr(train_ds, "class_weights", {})\n    qtd_classes = len(class_names) if class_names else NUM_CLASSES\n\n    print(f"\\n2. Instanciando a MobileNetV2 Híbrida para {qtd_classes} classes...")\n    model = build_custom_mobilenet(\n        num_classes=qtd_classes,\n        dropout_rate=DROPOUT_RATE,\n        weight_decay=WEIGHT_DECAY,\n    )\n    model.summary()\n\n    trainer = CustomTrainer(\n        model=model,\n        learning_rate=LEARNING_RATE,\n        class_weights=class_weights,\n        early_stopping_patience=EARLY_STOPPING_PATIENCE,\n        lr_patience=LR_PATIENCE,\n    )\n\n    print(f"\\n3A. [FASE 1 - WARMUP DA CABEÇA] Treinando camadas customizadas ({WARMUP_EPOCHS} épocas, LR={LEARNING_RATE})...")\n    trainer.fit(train_ds, val_ds, epochs=WARMUP_EPOCHS, stage_name="Fase 1: Warmup")\n    warmup_epochs_ran = len(trainer.history["train_loss"])\n\n    unfrozen = unfreeze_for_finetuning(model, unfreeze_top_layers=UNFREEZE_TOP_LAYERS)\n    if unfrozen > 0 and FINETUNE_EPOCHS > 0:\n        print(\n            f"\\n3B. [FASE 2 - FINE-TUNING] Ajustando {unfrozen} camadas superiores "\n            f"({FINETUNE_EPOCHS} épocas, LR={FINETUNE_LEARNING_RATE})..."\n        )\n        trainer.set_learning_rate(FINETUNE_LEARNING_RATE)\n        trainer.fit(train_ds, val_ds, epochs=FINETUNE_EPOCHS, stage_name="Fase 2: Fine-Tuning")\n\n    print("\\n4. Gerando Gráficos de Evolução e Matriz de Confusão por Classe...")\n    trainer.plot_training_curves("training_curves.png", warmup_epochs_ran=warmup_epochs_ran)\n    eval_report = trainer.evaluate_per_class(val_ds, class_names, "confusion_matrix.png")\n\n    print("\\n5. Exportando Modelo Quantizado (.tflite) e Metadados para Deploy (WhatsApp / AgriRAG)...")\n    tflite_file = export_quantized_tflite(model, "milho_pragas_model.tflite", quantize=True)\n\n    metadata = {\n        "model_file": str(tflite_file.name),\n        "size_kb": round(tflite_file.stat().st_size / 1024.0, 2),\n        "class_names": class_names,\n        "best_val_accuracy": round(float(trainer.best_val_acc), 4),\n        "best_val_loss": round(float(trainer.best_val_loss), 4),\n        "per_class_metrics": eval_report.get("per_class", {}),\n    }\n    Path("model_metadata.json").write_text(\n        json.dumps(metadata, ensure_ascii=False, indent=2),\n        encoding="utf-8",\n    )\n    print("✓ Metadados de produção salvos em \'model_metadata.json\'!")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('✓ Todos os 6 módulos Python aprimorados foram gravados na VM do Colab!')


### 4. Verificação / Coleta Rápida de Imagens das 4 Classes (`dataset/`)
Se as imagens já vieram junto com o `git clone` do repositório, **pula o download automaticamente**! Caso alguma classe esteja faltando, baixa em paralelo (24 threads + otimização `320x320`):

In [ ]:
from pathlib import Path

classes_esperadas = ['lagarta_cartucho', 'lagarta_espiga', 'vaquinha', 'milho_sadio']
faltantes = [
    c for c in classes_esperadas
    if not (Path('dataset') / c).exists() or len(list((Path('dataset') / c).glob('*.jpg'))) < 50
]

if not faltantes:
    print('✓ Todas as 4 classes já estão presentes na pasta dataset/ vindas do repositório! Nenhum download extra necessário.')
else:
    print(f'Baixando classes faltantes em paralelo ({faltantes})...')
    classes_arg = ' '.join(faltantes)
    !python 01_baixar_gbif.py --max-por-classe 600 --workers 24 --max-dim 320 --saida dataset --modo subpastas --incluir-nc --classes {classes_arg}

### 5. Complemento Opcional de Folhas de Milho Sadio (PlantVillage)
Adiciona folhas sadias extras do PlantVillage caso queira reforçar a classe `milho_sadio`:

In [ ]:
from pathlib import Path

sadio_dir = Path('dataset/milho_sadio')
qtd_sadio = len(list(sadio_dir.glob('*'))) if sadio_dir.exists() else 0
if qtd_sadio >= 300:
    print(f'✓ Classe milho_sadio já possui {qtd_sadio} imagens prontas no dataset!')
else:
    !git clone --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset.git /tmp/plantvillage
    !cd /tmp/plantvillage && git sparse-checkout set raw/color/Corn_\(maize\)___healthy
    !mkdir -p dataset/milho_sadio
    !cp -r /tmp/plantvillage/raw/color/Corn_\(maize\)___healthy/* dataset/milho_sadio/
    !echo "Fotos de milho_sadio disponíveis:" $(ls dataset/milho_sadio | wc -l)

### 6. Verificação das 4 Classes Oficiais, Data Flywheel (US03) e Pesos de Balanceamento (`class_weights`)

In [ ]:
import os
import sys
from pathlib import Path

for p in ['src', os.path.abspath('src')]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

from corn_pest_classifier.config import CLASS_NAMES
from corn_pest_classifier.data_loader import compute_class_weights, sync_flywheel_samples

dataset_path = Path('dataset')
print('=== VERIFICAÇÃO DO DATASET & BALANCEAMENTO (CULTURA: MILHO) ===')
sync_flywheel_samples(dataset_path)

classes_ativas = []
if dataset_path.exists():
    for c in CLASS_NAMES:
        pasta_classe = dataset_path / c
        if pasta_classe.exists() and pasta_classe.is_dir():
            imgs = [f for f in pasta_classe.glob('*') if f.is_file() and not f.name.endswith(('.txt', '.csv'))]
            print(f'  [OK] {c}: {len(imgs)} imagens')
            classes_ativas.append(c)
        else:
            print(f'  [PENDENTE] {c}: pasta não encontrada em dataset/{c}')

if classes_ativas:
    print()
    compute_class_weights(dataset_path, classes_ativas)

### 7. Visualização de Amostras & Data Augmentation Realista de Campo
Testa o pipeline `Field_Photo_Augmentation` (Flip, Rotação, Zoom, Contraste e Brilho simulando sol forte/sombra):

In [ ]:
!python src/corn_pest_classifier/visualizator.py

import os
from IPython.display import Image, display

if os.path.exists('preview_batch.png'):
    print('Amostras do lote original de treinamento:')
    display(Image('preview_batch.png'))

if os.path.exists('preview_augmentation.png'):
    print('Efeito das transformações de Data Augmentation de Campo:')
    display(Image('preview_augmentation.png'))

### 8. Treinamento em 2 Estágios (Warmup + Fine-Tuning), Avaliação por Classe e Exportação Quantizada (`.tflite`)
Executa o `main.py` com:
1. **Fase 1 (Warmup)**: Cabeça customizada + Bloco Residual Agrícola (`LR = 1e-3`).
2. **Fase 2 (Fine-Tuning)**: Descongelamento das 35 camadas superiores da MobileNetV2 (`LR = 1e-4`).
3. **Avaliação**: Gera `training_curves.png`, `confusion_matrix.png` e `model_metadata.json`.

In [ ]:
!python src/corn_pest_classifier/main.py

import os
from IPython.display import Image, display

if os.path.exists('training_curves.png'):
    print('\n📈 Curvas de Evolução do Treinamento (Warmup + Fine-Tuning):')
    display(Image('training_curves.png'))

if os.path.exists('confusion_matrix.png'):
    print('\n🎯 Matriz de Confusão na Validação:')
    display(Image('confusion_matrix.png'))

### 9. Teste de Inferência TFLite e Download dos Artefatos de Produção
Testa o arquivo `milho_pragas_model.tflite` recém-gerado com o `CornPestClassifier` e faz o download para plugar no `vision_node` do AgriRAG:

In [ ]:
import json
import os

if os.path.exists('model_metadata.json'):
    with open('model_metadata.json', encoding='utf-8') as f:
        print('=== METADADOS DO MODELO DE PRODUÇÃO ===')
        print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

try:
    from google.colab import files
    for artifact in ['milho_pragas_model.tflite', 'model_metadata.json']:
        if os.path.exists(artifact):
            print(f'Iniciando download de {artifact}...')
            files.download(artifact)
except ImportError:
    print('Ambiente local detectado: os arquivos já estão salvos na pasta do projeto.')